In [9]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql import functions as F 

from time import perf_counter

In [10]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [11]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("spark intro")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

In [12]:
print("Notebook Python:", sys.executable)
print("Spark worker Python:", spark.sparkContext.pythonExec)

Notebook Python: c:\Users\Lenovo\Desktop\UNIVR\AI MASTER\AI and cloud\ccdpp-pyspark-movielens\.venv\Scripts\python.exe
Spark worker Python: c:\Users\Lenovo\Desktop\UNIVR\AI MASTER\AI and cloud\ccdpp-pyspark-movielens\.venv\Scripts\python.exe


In [13]:
from pathlib import Path

import shutil
import urllib.request
import zipfile


PROJECT_DIR = Path.cwd().resolve()

DATA_DIR = PROJECT_DIR / "data"
ZIP_PATH = DATA_DIR / "ml-100k.zip"
DATASET_DIR = DATA_DIR / "ml-100k"

RESULTS_PATH = PROJECT_DIR / "als_results.csv"

DATASET_URL = (
    "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
)

DATA_DIR.mkdir(parents=True, exist_ok=True)

with urllib.request.urlopen(DATASET_URL) as response:
    with ZIP_PATH.open("wb") as output_file:
        shutil.copyfileobj(response, output_file)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(DATA_DIR)

# defining the input schema

In [14]:
filepath = "C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspark-movielens/data/ml-100k/u1.base"
ratings = spark.read.csv(filepath, sep="\t", inferSchema=True).toDF("user_id", "movie_id", "rating", "timestamp")
ratings.show(5)
ratings.printSchema()

+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|      1|       1|     5|874965758|
|      1|       2|     3|876893171|
|      1|       3|     4|878542960|
|      1|       4|     3|876893119|
|      1|       5|     3|889751712|
+-------+--------+------+---------+
only showing top 5 rows
root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)



In [15]:
test_filepath = "C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspark-movielens/data/ml-100k/u1.test"
test_ratings = spark.read.csv(test_filepath, sep="\t", inferSchema=True).toDF("user_id", "movie_id", "rating", "timestamp")
print(ratings.count())
print(test_ratings.count())

80000
20000


In [16]:
#possible rating values
ratings.select("rating").distinct().orderBy("rating").show()

+------+
|rating|
+------+
|     1|
|     2|
|     3|
|     4|
|     5|
+------+



In [17]:
# distinct users
ratings.select("user_id").distinct().count()

943

In [18]:
# distinct movies
ratings.select("movie_id").distinct().count()

1650

In [19]:
80000 / (943 * 1650)

0.051415533918185034

the matrix 943x1650 is very sparse
density = 5%
sparsity = 95%

In [20]:
train_users = ratings.select("user_id").distinct()
test_users = test_ratings.select("user_id").distinct()
# find test users absent from training using a left_anti join:
absent = test_users.join(train_users,"user_id", "left_anti")



In [21]:
print(absent.select("user_id").distinct().count())

0


In [22]:
train_movies = ratings.select("movie_id").distinct()
test_movies = test_ratings.select("movie_id").distinct()
absent_movies = test_movies.join(train_movies, "movie_id", "left_anti")
print(absent_movies.select("movie_id").distinct().count())

32


In [23]:
print(test_ratings.join(absent_movies, "movie_id", "inner").count())

32


In [24]:
print(test_ratings.join(absent_movies, "movie_id").groupBy("movie_id").count().show())

+--------+-----+
|movie_id|count|
+--------+-----+
|     857|    1|
|    1533|    1|
|    1561|    1|
|     830|    1|
|    1156|    1|
|    1493|    1|
|    1457|    1|
|    1582|    1|
|    1565|    1|
|     599|    1|
|    1505|    1|
|     711|    1|
|    1586|    1|
|    1458|    1|
|     852|    1|
|    1310|    1|
|    1520|    1|
|    1536|    1|
|    1498|    1|
|    1236|    1|
+--------+-----+
only showing top 20 rows
None


In [25]:
data_dir = "C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspark-movielens/data/ml-100k"
reg_eval = RegressionEvaluator(predictionCol="prediction", metricName="rmse", labelCol="rating")

folds = [1,2,3,4,5]
def foo(folds, configs) -> list: 
    results = []
    for fold in folds:
        X_train_path = str(f"{data_dir}/u{fold}.base")
        X_test_path = str(f"{data_dir}/u{fold}.test")
        X_train = (
            spark.read.csv(X_train_path, sep="\t", inferSchema=True)
            .toDF("user_id", "movie_id", "rating", "timestamp")
            .cache()
        )
        X_test = (
            spark.read.csv(X_test_path, sep="\t", inferSchema=True)
            .toDF("user_id", "movie_id", "rating", "timestamp")
            .cache()
        )
        X_train.count()
        X_test.count()
        for config in configs:
            start = perf_counter()
            estimator = ALS(
                maxIter=config["max_iter"],
                rank=config["rank"],
                regParam=config["reg_param"],
                userCol="user_id",
                itemCol="movie_id",
                ratingCol="rating",
                coldStartStrategy="drop",
                seed=42,
            )
            model = estimator.fit(X_train)

            end = perf_counter()
            training_time = end - start

            preds = model.transform(X_test)
            err = reg_eval.evaluate(preds)
            results.append({
                "model_name": config["model_name"],
                "fold": fold,
                "rank": config["rank"],
                "reg_param": config["reg_param"],
                "max_iter": config["max_iter"],
                "rmse": err,
                "training_time": training_time,
            })
        X_train.unpersist()
        X_test.unpersist()
    return results



summarize 5 folds with aggregation

In [26]:
ranks = [5,10,20]
reg_params = [0.05, 0.1, 0.15]
max_iters = [5,10]

configs = []

for rank in ranks:
    for reg_param in reg_params:
        for max_iter in max_iters:
            model_name = f"rank_{rank}_reg_param_{reg_param}_max_iter_{max_iter}"
            #create one configuration dictionary and append to configs
            config = {
                "model_name" : model_name,
                "rank": rank,
                "reg_param" : reg_param,
                "max_iter" : max_iter,
            }
            configs.append(config)

print(len(configs))
print(configs[0])
print(configs[-1])
len(set(config["model_name"] for config in configs))

18
{'model_name': 'rank_5_reg_param_0.05_max_iter_5', 'rank': 5, 'reg_param': 0.05, 'max_iter': 5}
{'model_name': 'rank_20_reg_param_0.15_max_iter_10', 'rank': 20, 'reg_param': 0.15, 'max_iter': 10}


18

In [27]:
grid_results = foo(folds, configs)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "c:\Users\Lenovo\Desktop\UNIVR\AI MASTER\AI and cloud\ccdpp-pyspark-movielens\.venv\Lib\site-packages\py4j\java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "c:\Users\Lenovo\Desktop\UNIVR\AI MASTER\AI and cloud\ccdpp-pyspark-movielens\.venv\Lib\site-packages\py4j\clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Lenovo\AppData\Local\Programs\Python\Python313\Lib\socket.py", line 719, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
import json

with open("als_cv_results.json", "w") as file:
    json.dump(grid_results, file, indent=2)

In [ ]:
from pyspark.sql import functions as F
grid_results_df = spark.createDataFrame(grid_results)

summary_df = (
    grid_results_df
    .groupBy("rank", "reg_param", "max_iter")
    .agg(
        F.avg("rmse").alias("mean_rmse"),
        F.stddev("rmse").alias("stddev_rmse"),
        F.avg("training_time").alias("training_time"),
        F.count("*").alias("n_fold")
    )
    .orderBy("mean_rmse")
)

In [ ]:
summary_df.show(truncate=False)

+----+---------+--------+------------------+---------------------+-----------------+------+
|rank|reg_param|max_iter|mean_rmse         |stddev_rmse          |training_time    |n_fold|
+----+---------+--------+------------------+---------------------+-----------------+------+
|5   |0.1      |10      |0.9279241644166085|0.005927733314254653 |8.37163303999696 |5     |
|10  |0.15     |10      |0.9282616531023906|0.0038271981841230085|7.543153699976392|5     |
|20  |0.15     |10      |0.9286828737797533|0.00413792354091356  |8.55022810001392 |5     |
|10  |0.1      |10      |0.9287676869126361|0.00512934851647727  |7.664241960039362|5     |
|20  |0.1      |10      |0.9302421052023313|0.004937770595499317 |8.133883520006203|5     |
|5   |0.15     |10      |0.9305390017579797|0.0036022792002274795|7.427018220024183|5     |
|10  |0.1      |5       |0.9353503215454342|0.004475350937893026 |4.547159760026261|5     |
|20  |0.1      |5       |0.9372358889166353|0.005723025818083815 |4.790652259974

: 

validation

In [ ]:
from pyspark.sql import functions as F
filepath = "C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspark-movielens/data/ml-100k/u5.base"
X_train = spark.read.csv(filepath, sep="\t", inferSchema=True).toDF("user_id", "movie_id", "rating", "timestamp")

number_of_ratings = X_train.groupBy("user_id").count().alias("counts")
number_of_ratings.orderBy(F.asc("count")).show(5)

+-------+-----+
|user_id|count|
+-------+-----+
|    873|    5|
|    895|    6|
|    941|    7|
|    917|    7|
|    876|    8|
+-------+-----+
only showing top 5 rows


assign a random position to every rating within each user, Then add the position

In [ ]:
from pyspark.sql.window import Window

user_window = Window.partitionBy("user_id").orderBy(F.rand(42))
ranked = X_train.withColumn(
    "row_num",
    F.row_number().over(user_window)
)
ranked.show(10)

+-------+--------+------+---------+-------+
|user_id|movie_id|rating|timestamp|row_num|
+-------+--------+------+---------+-------+
|      1|     119|     5|876893098|      1|
|      1|     203|     4|878542231|      2|
|      1|     221|     5|887431921|      3|
|      1|     258|     5|878873389|      4|
|      1|      22|     4|875072404|      5|
|      1|     124|     5|875071484|      6|
|      1|     251|     4|875071843|      7|
|      1|     138|     1|878543006|      8|
|      1|     103|     1|878542845|      9|
|      1|     144|     4|875073180|     10|
+-------+--------+------+---------+-------+
only showing top 10 rows


In [ ]:
ranked.filter(F.col("user_id")==873).show()

+-------+--------+------+---------+-------+
|user_id|movie_id|rating|timestamp|row_num|
+-------+--------+------+---------+-------+
|    873|     750|     3|891392303|      1|
|    873|     292|     5|891392177|      2|
|    873|     258|     3|891392818|      3|
|    873|     300|     4|891392238|      4|
|    873|     294|     4|891392303|      5|
+-------+--------+------+---------+-------+



we need each row to know how many total ratings that user has.

In [ ]:
count_window = Window.partitionBy("user_id")
cr = ranked.withColumn(
    "total",
    F.count("*").over(count_window)
)
cr.filter(F.col("user_id")==873).show()

+-------+--------+------+---------+-------+-----+
|user_id|movie_id|rating|timestamp|row_num|total|
+-------+--------+------+---------+-------+-----+
|    873|     750|     3|891392303|      1|    5|
|    873|     292|     5|891392177|      2|    5|
|    873|     258|     3|891392818|      3|    5|
|    873|     300|     4|891392238|      4|    5|
|    873|     294|     4|891392303|      5|    5|
+-------+--------+------+---------+-------+-----+



In [ ]:
cr = cr.withColumn(
    "val_count",
    F.floor(F.col("total")*0.2)
)
cr.filter(F.col("user_id")==873).show()

+-------+--------+------+---------+-------+-----+---------+
|user_id|movie_id|rating|timestamp|row_num|total|val_count|
+-------+--------+------+---------+-------+-----+---------+
|    873|     750|     3|891392303|      1|    5|        1|
|    873|     292|     5|891392177|      2|    5|        1|
|    873|     258|     3|891392818|      3|    5|        1|
|    873|     300|     4|891392238|      4|    5|        1|
|    873|     294|     4|891392303|      5|    5|        1|
+-------+--------+------+---------+-------+-----+---------+



In [ ]:
X_train.rdd.getNumPartitions()

1

In [28]:
filepath = "C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspark-movielens/data/ml-100k/u5.test"
X_test = spark.read.csv(filepath, sep="\t", inferSchema=True).toDF("user_id", "movie_id", "rating", "timestamp")

In [29]:
X_test.rdd.getNumPartitions()

1

In [30]:
X_train_2  = X_train.repartition(2)
X_train_2.rdd.getNumPartitions()

2

In [32]:
(X_train_2
 .withColumn("partition_id", F.spark_partition_id())
 .groupBy("partition_id")
 .count()
 .orderBy("partition_id")
 .show()
)

+------------+-----+
|partition_id|count|
+------------+-----+
|           0|40000|
|           1|40000|
+------------+-----+



In [ ]:
X_train_2.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- == Final Plan ==
   ResultQueryStage (5)
   +- ShuffleQueryStage (4), Statistics(sizeInBytes=3.1 MiB, rowCount=8.00E+4)
      +- Exchange (3)
         +- * Project (2)
            +- Scan csv  (1)
+- == Initial Plan ==
   Exchange (7)
   +- Project (6)
      +- Scan csv  (1)


(1) Scan csv 
Output [4]: [_c0#17, _c1#18, _c2#19, _c3#20]
Batched: false
Location: InMemoryFileIndex [file:/C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspark-movielens/data/ml-100k/u5.base]
ReadSchema: struct<_c0:int,_c1:int,_c2:int,_c3:int>

(2) Project [codegen id : 1]
Output [4]: [_c0#17 AS user_id#21, _c1#18 AS movie_id#22, _c2#19 AS rating#23, _c3#20 AS timestamp#24]
Input [4]: [_c0#17, _c1#18, _c2#19, _c3#20]

(3) Exchange
Input [4]: [user_id#21, movie_id#22, rating#23, timestamp#24]
Arguments: RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=3484]

(4) ShuffleQueryStage
Output [4]: [user_id#21, movie_id#22, rating#23, timestamp#24]
Argum

In [34]:
X_train_2.explain("simple")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=3484]
         +- *(1) Project [_c0#17 AS user_id#21, _c1#18 AS movie_id#22, _c2#19 AS rating#23, _c3#20 AS timestamp#24]
            +- FileScan csv [_c0#17,_c1#18,_c2#19,_c3#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspa..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<_c0:int,_c1:int,_c2:int,_c3:int>
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=3475]
   +- Project [_c0#17 AS user_id#21, _c1#18 AS movie_id#22, _c2#19 AS rating#23, _c3#20 AS timestamp#24]
      +- FileScan csv [_c0#17,_c1#18,_c2#19,_c3#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Lenovo/Desktop/UNIVR/

In [35]:
X_train_1 = X_train_2.coalesce(1)
X_train_1.explain("simple")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 1
   +- Exchange RoundRobinPartitioning(2), REPARTITION_BY_NUM, [plan_id=3596]
      +- Project [_c0#17 AS user_id#21, _c1#18 AS movie_id#22, _c2#19 AS rating#23, _c3#20 AS timestamp#24]
         +- FileScan csv [_c0#17,_c1#18,_c2#19,_c3#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspa..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<_c0:int,_c1:int,_c2:int,_c3:int>




In [36]:
X_train.coalesce(1).explain("simple")

== Physical Plan ==
Coalesce 1
+- *(1) Project [_c0#17 AS user_id#21, _c1#18 AS movie_id#22, _c2#19 AS rating#23, _c3#20 AS timestamp#24]
   +- FileScan csv [_c0#17,_c1#18,_c2#19,_c3#20] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspa..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<_c0:int,_c1:int,_c2:int,_c3:int>




In [40]:
print("default parallelism:", spark.sparkContext.defaultParallelism)
print("shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print(spark.conf.get("spark.sql.adaptive.enabled"))
grouped = X_train.groupBy("rating").count()
grouped.show()
grouped.rdd.getNumPartitions()

default parallelism: 2
shuffle partitions: 200
true
+------+-----+
|rating|count|
+------+-----+
|     1| 4952|
|     3|21668|
|     5|17033|
|     4|27354|
|     2| 8993|
+------+-----+



1

In [43]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
grouped_200 = X_train.groupBy("rating").count()
grouped_200.explain("simple")

== Physical Plan ==
*(2) HashAggregate(keys=[rating#23], functions=[count(1)])
+- Exchange hashpartitioning(rating#23, 200), ENSURE_REQUIREMENTS, [plan_id=3803]
   +- *(1) HashAggregate(keys=[rating#23], functions=[partial_count(1)])
      +- *(1) Project [_c2#19 AS rating#23]
         +- FileScan csv [_c2#19] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Lenovo/Desktop/UNIVR/AI MASTER/AI and cloud/ccdpp-pyspa..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<_c2:int>




In [44]:
grouped_200.show()

+------+-----+
|rating|count|
+------+-----+
|     1| 4952|
|     3|21668|
|     5|17033|
|     4|27354|
|     2| 8993|
+------+-----+



In [45]:
grouped_200.rdd.getNumPartitions()

200

In [46]:
grouped_with_pid = grouped_200.withColumn(
    "partition_id",
    F.spark_partition_id()
)

In [47]:
(
    grouped_with_pid
    .select("rating", "count", "partition_id")
    .orderBy("partition_id")
    .show()
)

+------+-----+------------+
|rating|count|partition_id|
+------+-----+------------+
|     1| 4952|          43|
|     3|21668|          51|
|     5|17033|          66|
|     4|27354|         102|
|     2| 8993|         174|
+------+-----+------------+



In [48]:
print(spark.sparkContext.uiWebUrl)

http://192.168.56.1:4040


In [49]:
grouped_200.collect()

[Row(rating=1, count=4952),
 Row(rating=3, count=21668),
 Row(rating=5, count=17033),
 Row(rating=4, count=27354),
 Row(rating=2, count=8993)]

In [50]:
X_train_cached = X_train.cache()
X_train_cached.count()

80000

In [58]:
spark.conf.set("spark.sql.shuffle.partitions", "2")

start = perf_counter()
X_train_cached.groupBy("rating").count().collect()
end = perf_counter()

print("2 partitions:", end - start)

2 partitions: 0.45046109999748296


In [59]:
spark.conf.set("spark.sql.shuffle.partitions", "200")


start = perf_counter()
X_train_cached.groupBy("rating").count().collect()
end = perf_counter()

print("200 partitions:", end - start)

200 partitions: 0.38412200000311714


In [60]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")
X_train_cached.unpersist()

DataFrame[user_id: int, movie_id: int, rating: int, timestamp: int]

In [66]:
als = ALS(
                maxIter=10,
                rank=5,
                regParam=0.1,
                userCol="user_id",
                itemCol="movie_id",
                ratingCol="rating",
                coldStartStrategy="drop",
                seed=42,
            )

print("user blocks:", als.getNumUserBlocks())
print("item blocks:", als.getNumItemBlocks())

user blocks: 10
item blocks: 10


In [ ]:
model = als.fit(X_train)
print("user-factor partitions:", model.userFactors.rdd.getNumPartitions())
print("item-factor partitions:", model.itemFactors.rdd.getNumPartitions())
model.setBlockSize

user-factor partitions: 10
item-factor partitions: 10


In [68]:
model.userFactors.show(3, truncate=False)
model.userFactors.printSchema()

+---+----------------------------------------------------------------+
|id |features                                                        |
+---+----------------------------------------------------------------+
|10 |[-1.183572, -1.8676832, -0.8637792, 0.063153595, -0.16644666]   |
|20 |[-0.90515435, -0.61506504, -1.439625, 0.55130404, 0.44561067]   |
|30 |[-1.3226473, -1.7412206, -0.87998486, -0.27543116, 0.0013214584]|
+---+----------------------------------------------------------------+
only showing top 3 rows
root
 |-- id: integer (nullable = false)
 |-- features: array (nullable = true)
 |    |-- element: float (containsNull = false)



In [81]:
block_configs = [
    (1, 1),
    (2, 2),
    (4, 4),
    (10, 10),
]
X_train = X_train.cache()
X_test = X_test.cache()

X_train.count()
X_test.count()

results = []
runs = [1,2,3]
for user_blocks, item_blocks in block_configs:
    for run in runs:
        als = ALS(
                        maxIter=10,
                        rank=5,
                        regParam=0.1,
                        userCol="user_id",
                        itemCol="movie_id",
                        ratingCol="rating",
                        coldStartStrategy="drop",
                        seed=42,
                        numUserBlocks=user_blocks,
                        numItemBlocks=item_blocks,
                    )
        start = perf_counter()
        model = als.fit(X_train)
        end = perf_counter()
        training_time = end -start
        pred = model.transform(X_test)
        rmse = RegressionEvaluator(predictionCol="prediction", metricName="rmse", labelCol="rating").evaluate(pred)
    
        result = {
            "user_blocks" : user_blocks,
            "item_blocks" : item_blocks,
            "training_time" : training_time,
            "rmse": rmse, 
            "run": run,
        }
        results.append(result)

   


In [83]:
df = spark.createDataFrame(results)
block_summary = (
    df
    .groupBy("user_blocks", "item_blocks")
    .agg(
        F.avg("rmse").alias("average rmse"),
        F.stddev("rmse").alias("rmse sttdev"),
        F.avg("training_time").alias("average training time"),
        F.stddev("training_time").alias("stddev training time")
    )
    .orderBy("average rmse")
)

block_summary.orderBy("average training time").show(truncate=False)

+-----------+-----------+------------------+-----------+---------------------+--------------------+
|user_blocks|item_blocks|average rmse      |rmse sttdev|average training time|stddev training time|
+-----------+-----------+------------------+-----------+---------------------+--------------------+
|1          |1          |0.9282650930020032|0.0        |1.9174823333354045   |0.3555262024931061  |
|2          |2          |0.9229506584936907|0.0        |1.922758633333918    |0.11353339246132538 |
|4          |4          |0.9218500877164053|0.0        |2.3907268999998146   |0.057135293069085547|
|10         |10         |0.9325057141147263|0.0        |7.223281333332125    |1.0445416500204732  |
+-----------+-----------+------------------+-----------+---------------------+--------------------+



scaling with cores

In [ ]:

spark.stop()
def create_spark_session(num_cores):
    new_spark = (
    SparkSession.builder
    .master(f"local[{num_cores}]")
    .appName("spark intro")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
    )

    return new_spark

spark = create_spark_session(1)
print("requested cores:", 1)
print("default parallelism:", spark.sparkContext.defaultParallelism)  

requested cores: 1
default parallelism: 1


In [87]:
def load_data(spark):
    train_df = (
        spark.read
        .option("sep", "\t")
        .schema("user_id INT, movie_id INT, rating DOUBLE, timestamp LONG")
        .csv("data/ml-100k/u5.base")
    )
    test_df = (
        spark.read
        .option("sep", "\t")
        .schema("user_id INT, movie_id INT, rating DOUBLE, timestamp LONG")
        .csv("data/ml-100k/u5.test")
    )
    train_df.cache()
    test_df.cache()

    train_df.count()
    test_df.count()
    return train_df, test_df

In [88]:


core_results = []
core_values = [1, 2, 4]
for num_cores in core_values:
    spark.stop()
    spark = create_spark_session(num_cores)

    X_train, X_test = load_data(spark)

    # warm-up fit
    als = ALS(
                                        maxIter=10,
                                        rank=5,
                                        regParam=0.1,
                                        userCol="user_id",
                                        itemCol="movie_id",
                                        ratingCol="rating",
                                        coldStartStrategy="drop",
                                        seed=42,
                                        numUserBlocks=4,
                                        numItemBlocks=4,
                                    )
    warmup_model = als.fit(X_train)

    reg_eval = RegressionEvaluator(
    predictionCol="prediction",
    metricName="rmse",
    labelCol="rating",
    )

    for run in range(1, 4):

        # time only fit()
        start = perf_counter()
        model = als.fit(X_train)
        end = perf_counter()
        training_time = end - start
        # predict and evaluate
        pred = model.transform(X_test)
        err = reg_eval.evaluate(pred)
        # append a result dictionary
        result = {
            "run": run,
            "num_cores": num_cores,
            "training_time": training_time,
            "rmse": err,
            "rank" : 5,
            "user_blocks": 4,
            "item_blocks": 4,
        }
        core_results.append(result)

    X_train.unpersist()
    X_test.unpersist()

In [90]:
print(len(core_results))

df = spark.createDataFrame(core_results)

core_summary = (
    df
    .groupBy("num_cores")
    .agg(
        F.avg("rmse").alias("mean_rmse"),
        F.stddev("rmse").alias("std_rmse"),
        F.avg("training_time").alias("mean_time"),
        F.stddev("training_time").alias("std_time"),
    )
)

9


In [94]:
core_summary.show()

+---------+------------------+--------+------------------+------------------+
|num_cores|         mean_rmse|std_rmse|         mean_time|          std_time|
+---------+------------------+--------+------------------+------------------+
|        1|0.9218500877164053|     0.0| 4.019634100002198| 0.365820467274788|
|        2|0.9218500877164053|     0.0|3.1672436999967126|0.7089747525347719|
|        4|0.9218500877164053|     0.0|2.4309414333329187|0.4914601598425518|
+---------+------------------+--------+------------------+------------------+



calculate speedup and efficiency

In [97]:
baseline_time = (
    core_summary
    .filter(F.col("num_cores")==1)
    .select("mean_time")
    .first()[0]
)

In [92]:
baseline_time

4.019634100002198

In [98]:
core_summary =(
    core_summary
    .withColumn
    (
    "speedup",
    F.lit(baseline_time)/F.col("mean_time")
    )
    .withColumn
    (
        "effciciency",
        F.col("speedup")/F.col("num_cores")
    )
)

core_summary.show()

+---------+------------------+--------+------------------+------------------+------------------+------------------+
|num_cores|         mean_rmse|std_rmse|         mean_time|          std_time|           speedup|       effciciency|
+---------+------------------+--------+------------------+------------------+------------------+------------------+
|        1|0.9218500877164053|     0.0| 4.019634100002198| 0.365820467274788|               1.0|               1.0|
|        2|0.9218500877164053|     0.0|3.1672436999967126|0.7089747525347719| 1.269126875208993|0.6345634376044965|
|        4|0.9218500877164053|     0.0|2.4309414333329187|0.4914601598425518|1.6535297991490965|0.4133824497872741|
+---------+------------------+--------+------------------+------------------+------------------+------------------+



In [103]:
fractions = [0.25, 0.50, 1.00]
size_results = []
for fraction in fractions:

    spark.stop()
    spark = create_spark_session(4)

    X_train, X_test = load_data(spark)
    X_train_subset = (
        X_train.sample(
            fraction=fraction,
            seed=42,
            withReplacement=False
        )
        .cache()
    )
    num_ratings = X_train_subset.count()
    test_count = X_test.count()

    # warm-up fit
    als = ALS(
                maxIter=10,
                rank=5,
                regParam=0.1,
                userCol="user_id",
                itemCol="movie_id",
                ratingCol="rating",
                coldStartStrategy="drop",
                seed=42,
                numUserBlocks=4,
                numItemBlocks=4,
            )
    warmup_model = als.fit(X_train_subset)

    reg_eval = RegressionEvaluator(
        predictionCol="prediction",
        metricName="rmse",
        labelCol="rating",
    )

    for run in range(1, 4):
    
            # time only fit()
            start = perf_counter()
            model = als.fit(X_train_subset)
            end = perf_counter()
            training_time = end - start
            # predict and evaluate
            pred = model.transform(X_test)
            err = reg_eval.evaluate(pred)
            prediction_count = pred.count()
            coverage = prediction_count / test_count
            err = reg_eval.evaluate(pred)

            pred.unpersist()
            # append a result dictionary
            result = {
                "run": run,
                "num_ratings": num_ratings,
                "training_time": training_time,
                "rmse": err,
                "rank" : 5,
                "subset_fraction": fraction,
                "num_cores": 4,
                "coverage": coverage,
                "prediction_count": prediction_count,
            }
            size_results.append(result)
    
    X_train_subset.unpersist()
    X_test.unpersist()

In [104]:
size_results_df = spark.createDataFrame(size_results)
size_results_summary = (
    size_results_df
    .groupBy("subset_fraction", "num_ratings")
    .agg(
        F.avg("training_time").alias("mean_time"),
        F.stddev("training_time").alias("std_time"),
        F.avg("rmse").alias("mean_rmse"),
        F.avg("coverage").alias("mean_coverage"),
    )
    .orderBy("num_ratings")
)

size_results_summary.show(truncate=False)

+---------------+-----------+------------------+--------------------+------------------+------------------+
|subset_fraction|num_ratings|mean_time         |std_time            |mean_rmse         |mean_coverage     |
+---------------+-----------+------------------+--------------------+------------------+------------------+
|0.25           |20068      |2.105966600002527 |0.2149681057062694  |1.1100717136478657|0.9882            |
|0.5            |40065      |1.890527600000496 |0.030143701756758488|0.9882942835797267|0.99545           |
|1.0            |80000      |1.8504780666650429|0.04539806259707315 |0.9218500877164053|0.9982000000000001|
+---------------+-----------+------------------+--------------------+------------------+------------------+

